# UFM-Rec — huấn luyện trên Colab trong trình duyệt

**Tệp → Mở sổ tay → Tải lên** notebook này, chọn runtime **T4** (hoặc **L4** nếu tài khoản có, rồi đổi `GPU_TYPE` ở cell đầu), rồi chọn **Thời gian chạy → Chạy tất cả**. Đăng nhập/gắn Drive khi Colab yêu cầu.

Nếu chưa có dữ liệu trên Drive, cell dữ liệu sẽ mở nút **Choose Files**. Mỗi lần chọn **một** ZIP được yêu cầu trong thư mục máy tính:
`D:\Nam3_ki1\UFM-Rec_Backups\colab_training_inputs_20261029`

Cần sáu tệp `bundle_000.zip` đến `bundle_005.zip` (tổng khoảng 3,33 GiB). ZIP đã nhận được kiểm tra và lưu lên Drive; phiên sau không cần tải lại. Nếu muốn tải bằng Google Drive trước, đặt sáu ZIP vào `MyDrive/UFM-Rec_Experiments/ufm_followup_20261028/inputs` rồi chạy notebook. Không tải lại bộ Amazon gốc hay tạo lại CLIP.

Notebook bắt đầu **full UFM / seed 42** trên dữ liệu thật, lưu checkpoint sau mỗi đoạn, rồi resume đúng vị trí. Giữ đủ giao thức 15 lượt qua nhiều phiên; một phiên Colab có thể chỉ hoàn thành một phần lượt đầu. Chỉ báo hoàn tất khi có `completed.json` hợp lệ và kết quả validation.

Source khóa ở commit `4b72a2de05e16a14a068b26389bdf3d45e2b8264`. Không đổi seed/epoch và không chọn mô hình bằng test.


In [ ]:
from pathlib import Path
import sys, os, json, time, math, hashlib, shutil, subprocess, importlib.util

SOURCE_COMMIT = '4b72a2de05e16a14a068b26389bdf3d45e2b8264'
SOURCE = Path('/content/UFM-Rec')
WORKSPACE = Path('/content/ufm_workspace')
GPU_TYPE = 'T4'  # đổi thành 'L4' nếu chạy runtime L4; phép đo throughput lưu riêng theo GPU
DRIVE_ROOT = Path('/content/drive/MyDrive/UFM-Rec_Experiments/ufm_followup_20261028')
SESSION_BUDGET_MINUTES = 120
SESSION_T0 = time.monotonic()
SLICE_STEPS = 1000
PROBE_STEPS = 100
# Keep all 15 locked jobs across sessions; never silently reduce the protocol.
CONTINUE_LOCKED_PLAN_ACROSS_SESSIONS = True

if os.name == 'nt' or not Path('/content').is_dir():
    raise RuntimeError(f'Mở notebook bằng Google Colab và chọn runtime {GPU_TYPE}.')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError(f'Chọn Thời gian chạy > Thay đổi loại thời gian chạy > {GPU_TYPE} GPU, rồi chạy lại.')
gpu_line = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.free,utilization.gpu','--format=csv,noheader'], text=True).strip()
print('GPU:', gpu_line)
if GPU_TYPE not in gpu_line:
    raise RuntimeError(f'Cần kernel {GPU_TYPE} để phép đo đúng thiết bị đã chọn.')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--branch','followup/multimodal-20261028',
                    'https://github.com/anky06-ky/UFM-Rec.git',str(SOURCE)],check=True)
    subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(SOURCE),'checkout','--detach',SOURCE_COMMIT],check=True)
actual = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
if actual != SOURCE_COMMIT:
    raise RuntimeError(f'Source commit mismatch: {actual}; checkout preserved for inspection.')
# Preserve the CUDA torch provided by Colab; pin numpy for strict resume.
subprocess.run([sys.executable,'-m','pip','install','--quiet','numpy==2.2.6','scipy==1.17.1'],check=True)
RUNTIME_VERSIONS=json.loads(subprocess.check_output([sys.executable,'-c',
    "import torch,numpy,scipy,json; assert torch.cuda.is_available(); print(json.dumps(dict(torch=str(torch.__version__),numpy=numpy.__version__,scipy=scipy.__version__)))"],text=True))
print(RUNTIME_VERSIONS)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''): h.update(block)
    return h.hexdigest()
def save_json(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf-8'); tmp.replace(path)
print('SOURCE_OK',actual)

## Gắn Drive

Hoàn tất đăng nhập/cấp quyền trực tiếp trong trình duyệt nếu được yêu cầu. Chỉ xác nhận checkpoint đã sao lưu sau khi bản Drive khớp SHA. Thư mục thí nghiệm tách riêng trên Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive chưa gắn thành công; chưa chạy train.')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
probe=DRIVE_ROOT/'write_test.tmp'; probe.write_text('ok'); probe.replace(DRIVE_ROOT/'write_test.txt')
print('DRIVE_WRITE_OK',DRIVE_ROOT)
version_file=DRIVE_ROOT/'runtime_versions.json'
if version_file.exists() and json.loads(version_file.read_text())!=RUNTIME_VERSIONS:
    raise RuntimeError('Phiên bản torch/numpy/scipy khác lượt trước. Chọn lại runtime 2026.07 đúng môi trường trước khi resume.')
save_json(version_file,RUNTIME_VERSIONS)


## Nhận dữ liệu một lần, dùng lại từ Drive ở các phiên sau

Nếu cell hiện nút **Choose Files**, chọn đúng **một ZIP** trong danh sách còn thiếu. Đợi notebook xác minh và lưu lên Drive trước khi chọn gói tiếp theo. Gói đã có trên Drive hoặc `/content` được dùng lại tự động. Manifest đã nhúng trong notebook, không cần chọn tệp JSON.

Chỉ bắt đầu train sau khi hiện `COLAB_PREFLIGHT_PASSED 9`.


In [ ]:
BACKUP_MANIFEST_TEXT = '{\r\n  "schema_version": 1,\r\n  "created_at_utc": "2026-10-29T00:08:13.258655+00:00",\r\n  "source_root": "D:\\\\Nam3_ki1\\\\UFM-Rec_Backups\\\\restored_complete_20261028",\r\n  "phase": "complete",\r\n  "files": [\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/complete.json",\r\n      "bytes": 299,\r\n      "sha256": "e48c666dd4e782026558bd9bf1156be22f9a4d4618e843278e5f18bd2313fd2c",\r\n      "mtime_ns": 1793161985241086200,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/complete.json"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/histories.npy",\r\n      "bytes": 338467968,\r\n      "sha256": "d19d133bb0b508b9caf61767d3008394b2a5b144c973f5317d63b4029cc20042",\r\n      "mtime_ns": 1793162001938237400,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/histories.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/targets.npy",\r\n      "bytes": 16923520,\r\n      "sha256": "d54ed27ca6390de5e28b154089bc77a34f4e7d556f0616a21ab204d4a3cc00b2",\r\n      "mtime_ns": 1793162005046363800,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/targets.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/term_ids.npy",\r\n      "bytes": 196363904,\r\n      "sha256": "6cfc85ea09398116d49183ca713d3949dcf4bfba5aa9cf025c299d081c9e5021",\r\n      "mtime_ns": 1793162007812209100,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/term_ids.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/term_weights.npy",\r\n      "bytes": 196363904,\r\n      "sha256": "d10c5c25c7efe39b339e67bc1649dcee0f82bbfa53d7082d2a8b4da6264fbe37",\r\n      "mtime_ns": 1793162012009323300,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/term_weights.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/train_counts.npy",\r\n      "bytes": 6136496,\r\n      "sha256": "b8c37d48e55654eb091d3d8e9a3a61efa76c2e2c988747fbecd1b16f536f9564",\r\n      "mtime_ns": 1793162013386260700,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/train_counts.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/gpu_cache_h20_t64_v1/users.npy",\r\n      "bytes": 33846912,\r\n      "sha256": "b540626d47b56ad0745817714dab11a77521fa8c58c37b294132f8789a8c2025",\r\n      "mtime_ns": 1793162013741220300,\r\n      "canonical": "data/processed/gpu_cache_h20_t64_v1/users.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/content/content_report.json",\r\n      "bytes": 1397,\r\n      "sha256": "fd18d912da3de15d39461694ef2fa26d6f33a10b9ecc5e40dade3c216ec867cf",\r\n      "mtime_ns": 1793161985252598700,\r\n      "canonical": "data/processed/toys_games_full_temporal/content/content_report.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/content/items.csv.gz",\r\n      "bytes": 7515509,\r\n      "sha256": "e67e6310a6076f1e66ff14b8ce68e3d28a1aba204498175ae77eb0b04d0050fc",\r\n      "mtime_ns": 1793162014019636900,\r\n      "canonical": "data/processed/toys_games_full_temporal/content/items.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/content/products_text.csv.gz",\r\n      "bytes": 296595579,\r\n      "sha256": "5e134cfd3ac4e94e78b3bdac9bbd40927fb57fbfa45874e57fceae7abe5aca13",\r\n      "mtime_ns": 1793162015921069600,\r\n      "canonical": "data/processed/toys_games_full_temporal/content/products_text.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/content/tfidf_all.npz",\r\n      "bytes": 532281300,\r\n      "sha256": "410981218522b8532963e82fadee6d434bc6235b0a7af5a7d0ae14028d0ba4d3",\r\n      "mtime_ns": 1793162022776623000,\r\n      "canonical": "data/processed/toys_games_full_temporal/content/tfidf_all.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/content/vectorizer.joblib",\r\n      "bytes": 246229,\r\n      "sha256": "4d5071ee2c83a0bd84daa6965654d0993c6cae8ea5acd32dc376d5d69cea187f",\r\n      "mtime_ns": 1793161985263135400,\r\n      "canonical": "data/processed/toys_games_full_temporal/content/vectorizer.joblib"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation/samples_test.csv.gz",\r\n      "bytes": 2115628,\r\n      "sha256": "7e4510cebdc5b1d355f8ee9d9aabc7fc334c6daaf4df815a0787195500108bbf",\r\n      "mtime_ns": 1793162026388777100,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation/samples_test.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation/samples_test.npz",\r\n      "bytes": 12784303,\r\n      "sha256": "cda970073ba195bb092c515e29ab9e28812460d034392fc4d91711e68a14b5da",\r\n      "mtime_ns": 1793162026536254400,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation/samples_test.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation/samples_validation.csv.gz",\r\n      "bytes": 2108847,\r\n      "sha256": "eaceba64786421e274fc800938b1015318d6a806c42e116405a6f23ec5d96eb2",\r\n      "mtime_ns": 1793162026637251800,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation/samples_validation.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation/samples_validation.npz",\r\n      "bytes": 12776088,\r\n      "sha256": "8efa116508fddd6b5227b69d6c65e3b635de634dd542046463fda84e78b47162",\r\n      "mtime_ns": 1793162026758870800,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation/samples_validation.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation/sampling_report.json",\r\n      "bytes": 1329,\r\n      "sha256": "5ee425bd39bfb0dcd68a7b94ba16a7004e457ff73afc2ad7b79698f4fde1abd1",\r\n      "mtime_ns": 1793161985276846700,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation/sampling_report.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.csv.gz",\r\n      "bytes": 2117629,\r\n      "sha256": "273501122f37028c607fbe3c4a699d5c66e073f1f66d1210c4244de831178521",\r\n      "mtime_ns": 1793162026857139800,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.npz",\r\n      "bytes": 12408599,\r\n      "sha256": "a4422c2cfe420e7ffc072915b16ff88a218d640054fb107c937d867dca20b109",\r\n      "mtime_ns": 1793162026969582100,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.csv.gz",\r\n      "bytes": 2110935,\r\n      "sha256": "d529b0aeb68e4b026556756601efc4380b179dc43f96e05a76eccef0843a051c",\r\n      "mtime_ns": 1793162027067898300,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.npz",\r\n      "bytes": 12355653,\r\n      "sha256": "c71906e65ce4067b405a94c13e22e8a25a17f46cbee5b1315f07d01d4d1e59df",\r\n      "mtime_ns": 1793162027176568000,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/evaluation_v2/sampling_report.json",\r\n      "bytes": 1388,\r\n      "sha256": "f8778b5bc6915499b13efc15b36ffdafa272b999c16a861dccdd33b563a45afc",\r\n      "mtime_ns": 1793162273255081100,\r\n      "canonical": "data/processed/toys_games_full_temporal/evaluation_v2/sampling_report.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog.csv.gz",\r\n      "bytes": 14377006,\r\n      "sha256": "65d63e7b58babb133db78f3369b3224db6a451d80c4f616a4b8c0023f73b9003",\r\n      "mtime_ns": 1793162027342654800,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog_report.json",\r\n      "bytes": 1788,\r\n      "sha256": "d7ad84e43588a4aee0b8a26b62cef2d92d4e83c821182e94920d86d9c1b7503f",\r\n      "mtime_ns": 1793161985288394900,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog_report.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/complete.json",\r\n      "bytes": 1677,\r\n      "sha256": "c50d8a06647390bb44e433ad4d49bd606ee547a39ff959978ce1ada34ff4db98",\r\n      "mtime_ns": 1793162273264409900,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/complete.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image.npy",\r\n      "bytes": 785455232,\r\n      "sha256": "eece460c4860751f9463bbeaac64fe09cc939f44e6f96d7435d5239faf155efa",\r\n      "mtime_ns": 1793162037974391400,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image_errors.jsonl",\r\n      "bytes": 626529,\r\n      "sha256": "7d40aadcc09410f937e7d1bb28f740b184e4d1b2035f716de2a07db4e5e07c7e",\r\n      "mtime_ns": 1793161985302630900,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image_errors.jsonl"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/modalities.npy",\r\n      "bytes": 1534220,\r\n      "sha256": "03f495e9429751c05ebf96618cf70546e1d91b4a335a77017adee62eb87f07c2",\r\n      "mtime_ns": 1793162043490906200,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/modalities.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/progress.json",\r\n      "bytes": 958,\r\n      "sha256": "c89d1ab0dde190b1908d34036a96bcb574c9c3514a505f60414f4b611c4c12b6",\r\n      "mtime_ns": 1793161985320509300,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/progress.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/text.npy",\r\n      "bytes": 785455232,\r\n      "sha256": "d8f507d65712fefdd98c5463d55836512ea281d8ecf199508e595c0ba86f1e3d",\r\n      "mtime_ns": 1793162054106161300,\r\n      "canonical": "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/text.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/interactions_test.csv.gz",\r\n      "bytes": 31580575,\r\n      "sha256": "d199de6a17e7c2ac694c0831cfd5937dd551f29bff1dc440eb80672aa2794c55",\r\n      "mtime_ns": 1793162060519716300,\r\n      "canonical": "data/processed/toys_games_full_temporal/interactions_test.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/interactions_train.csv.gz",\r\n      "bytes": 224904933,\r\n      "sha256": "2f66eb960861c0477ee2a590f341b8082598ef7a9578da064c796f34a594ba46",\r\n      "mtime_ns": 1793162062058132400,\r\n      "canonical": "data/processed/toys_games_full_temporal/interactions_train.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/interactions_validation.csv.gz",\r\n      "bytes": 32458877,\r\n      "sha256": "8ab1cf34b4d0097c52fbbf2d267772edc1b84ce1fd9d34526db016ab05850aac",\r\n      "mtime_ns": 1793162063968541300,\r\n      "canonical": "data/processed/toys_games_full_temporal/interactions_validation.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/item_statistics.csv.gz",\r\n      "bytes": 4672968,\r\n      "sha256": "1133a38e307717cdea9fdad1918e0a18afc794448af5743efdd80c67bb64f6da",\r\n      "mtime_ns": 1793162064275492100,\r\n      "canonical": "data/processed/toys_games_full_temporal/item_statistics.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/models/content_baseline/ranks_validation.npz",\r\n      "bytes": 84386,\r\n      "sha256": "74530448eb70e699ca25f84966a476e1ef2bdd8c211d3f58d64ff155498e5ad2",\r\n      "mtime_ns": 1793161985334793500,\r\n      "canonical": "data/processed/toys_games_full_temporal/models/content_baseline/ranks_validation.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/models/content_baseline_v2/metrics.json",\r\n      "bytes": 7469,\r\n      "sha256": "f3d242294836ad4a6d8e83c2de50b825a368f0d91f4b1c2e1eb11d5d392bb2cc",\r\n      "mtime_ns": 1793161985348292400,\r\n      "canonical": "data/processed/toys_games_full_temporal/models/content_baseline_v2/metrics.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_test.npz",\r\n      "bytes": 93572,\r\n      "sha256": "1854b36985ff4237a1f5dff2f162756ff1fd7d5ed7225c75f4e59bb9f908b6ab",\r\n      "mtime_ns": 1793161985360384900,\r\n      "canonical": "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_test.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_validation.npz",\r\n      "bytes": 93352,\r\n      "sha256": "3300ef6a93725b71f74c5f0d7c0f39ad6d62f40ff34cf5a5d9880ee2623fa60d",\r\n      "mtime_ns": 1793161985368637800,\r\n      "canonical": "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_validation.npz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/split_manifest.json",\r\n      "bytes": 2400,\r\n      "sha256": "585f102f14e12bfae332f53d28bb1f853acde1586eb102555c4b118ccee75763",\r\n      "mtime_ns": 1793161985382435700,\r\n      "canonical": "data/processed/toys_games_full_temporal/split_manifest.json"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/test_with_history.csv.gz",\r\n      "bytes": 42243500,\r\n      "sha256": "00d78ed12a9d649ed9a525110af7800807980dd9badacf6ee6f64fe944365ee7",\r\n      "mtime_ns": 1793162064641106300,\r\n      "canonical": "data/processed/toys_games_full_temporal/test_with_history.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/train_with_history.csv.gz",\r\n      "bytes": 131696643,\r\n      "sha256": "8e4f5bb03a57ee4407913fb7de4c6f9c27f768edee0dc5c66cc67976a05c7b7f",\r\n      "mtime_ns": 1793162066787125800,\r\n      "canonical": "data/processed/toys_games_full_temporal/train_with_history.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/toys_games_full_temporal/validation_with_history.csv.gz",\r\n      "bytes": 43828507,\r\n      "sha256": "2845d3d9497a4498474cf0f19c00dc2dedf488433dbdf6afc5fc4e80522fafcd",\r\n      "mtime_ns": 1793162067920385000,\r\n      "canonical": "data/processed/toys_games_full_temporal/validation_with_history.csv.gz"\r\n    },\r\n    {\r\n      "path": "data/processed/ufm_positive_graph_h20_v1/complete.json",\r\n      "bytes": 1075,\r\n      "sha256": "db655f42ff829de69d3c4b0c5e0f0f1b33e00aae84a2fb1a6b406e6404f1b9b4",\r\n      "mtime_ns": 1793161985396544500,\r\n      "canonical": "data/processed/ufm_positive_graph_h20_v1/complete.json"\r\n    },\r\n    {\r\n      "path": "data/processed/ufm_positive_graph_h20_v1/positive_indptr.npy",\r\n      "bytes": 13011504,\r\n      "sha256": "2d82da64f8ec956ab81ed6d8c96ff86a5c5dcc7f694b6257f43fe2fd57e781a4",\r\n      "mtime_ns": 1793162068317807800,\r\n      "canonical": "data/processed/ufm_positive_graph_h20_v1/positive_indptr.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/ufm_positive_graph_h20_v1/positive_items.npy",\r\n      "bytes": 23344008,\r\n      "sha256": "e7e21bcf53834655d4b73a5d9c8d45b45ce918c0c4d1bc4613a191ebd561f730",\r\n      "mtime_ns": 1793162068751750400,\r\n      "canonical": "data/processed/ufm_positive_graph_h20_v1/positive_items.npy"\r\n    },\r\n    {\r\n      "path": "data/processed/ufm_positive_graph_h20_v1/user_keys.npy",\r\n      "bytes": 13011496,\r\n      "sha256": "887a78047813afd32dbf8731f333cfc111fe31c8be24c0dd67881dfe98210ba7",\r\n      "mtime_ns": 1793162069091855200,\r\n      "canonical": "data/processed/ufm_positive_graph_h20_v1/user_keys.npy"\r\n    },\r\n    {\r\n      "path": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.csv.gz",\r\n      "bytes": 2110935,\r\n      "sha256": "f82d61588a1dfc3f7c55d1611dd3f3466b79524ae80f18a2b82b026570fc6da4",\r\n      "mtime_ns": 1793162274960514500,\r\n      "canonical": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.csv.gz"\r\n    },\r\n    {\r\n      "path": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.npz",\r\n      "bytes": 12355654,\r\n      "sha256": "9387e66e9f1905c44d02b65c3809f5fa8b6c022f6abf0c1a37d52cb57767a60e",\r\n      "mtime_ns": 1793162274990612500,\r\n      "canonical": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.npz"\r\n    },\r\n    {\r\n      "path": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/sampling_report.json",\r\n      "bytes": 1048,\r\n      "sha256": "174999911397aad64096b47be738a0ad2a15b2b91a3cb127e3e7216fbd235aa2",\r\n      "mtime_ns": 1793162273449012300,\r\n      "canonical": "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/sampling_report.json"\r\n    }\r\n  ],\r\n  "bundles": [\r\n    {\r\n      "path": "bundle_000.zip",\r\n      "bytes": 380565750,\r\n      "sha256": "19c38831cd0d67da9e4c5b0260cef543829aa72f37c0b45741ce5cac0b9a5e8e",\r\n      "files": [\r\n        "data/processed/gpu_cache_h20_t64_v1/complete.json",\r\n        "data/processed/gpu_cache_h20_t64_v1/histories.npy",\r\n        "data/processed/gpu_cache_h20_t64_v1/targets.npy",\r\n        "data/processed/gpu_cache_h20_t64_v1/term_ids.npy",\r\n        "data/processed/gpu_cache_h20_t64_v1/term_weights.npy",\r\n        "data/processed/gpu_cache_h20_t64_v1/train_counts.npy",\r\n        "data/processed/gpu_cache_h20_t64_v1/users.npy",\r\n        "data/processed/toys_games_full_temporal/content/content_report.json",\r\n        "data/processed/toys_games_full_temporal/content/items.csv.gz"\r\n      ]\r\n    },\r\n    {\r\n      "path": "bundle_001.zip",\r\n      "bytes": 312981624,\r\n      "sha256": "0ead4ecac781fde7ead1fbea96f24987ad06a3ebdadea596fa4e4f35db86d768",\r\n      "files": [\r\n        "data/processed/toys_games_full_temporal/content/products_text.csv.gz"\r\n      ]\r\n    },\r\n    {\r\n      "path": "bundle_002.zip",\r\n      "bytes": 641218875,\r\n      "sha256": "9e0ad912ec4456aba40c666f7ff53b185f7d47e45ca963c440efff813e4582b4",\r\n      "files": [\r\n        "data/processed/toys_games_full_temporal/content/tfidf_all.npz",\r\n        "data/processed/toys_games_full_temporal/content/vectorizer.joblib",\r\n        "data/processed/toys_games_full_temporal/evaluation/samples_test.csv.gz",\r\n        "data/processed/toys_games_full_temporal/evaluation/samples_test.npz",\r\n        "data/processed/toys_games_full_temporal/evaluation/samples_validation.csv.gz",\r\n        "data/processed/toys_games_full_temporal/evaluation/samples_validation.npz",\r\n        "data/processed/toys_games_full_temporal/evaluation/sampling_report.json",\r\n        "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.csv.gz",\r\n        "data/processed/toys_games_full_temporal/evaluation_v2/samples_test.npz",\r\n        "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.csv.gz",\r\n        "data/processed/toys_games_full_temporal/evaluation_v2/samples_validation.npz",\r\n        "data/processed/toys_games_full_temporal/evaluation_v2/sampling_report.json",\r\n        "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog.csv.gz",\r\n        "data/processed/toys_games_full_temporal/foundation_catalog_v1/catalog_report.json",\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/complete.json"\r\n      ]\r\n    },\r\n    {\r\n      "path": "bundle_003.zip",\r\n      "bytes": 820363558,\r\n      "sha256": "8e0fff7d68045a2b31685a021d57c529817d500e09f9b05ee997dc4a29bdc18f",\r\n      "files": [\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image.npy",\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/image_errors.jsonl",\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/modalities.npy",\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/progress.json"\r\n      ]\r\n    },\r\n    {\r\n      "path": "bundle_004.zip",\r\n      "bytes": 830255301,\r\n      "sha256": "d7dd5f87003bfb8513f8e483f8aae8262f384cac11c144e57eebb037a8bffd5b",\r\n      "files": [\r\n        "data/processed/toys_games_full_temporal/foundation_clip_b32_v1/text.npy"\r\n      ]\r\n    },\r\n    {\r\n      "path": "bundle_005.zip",\r\n      "bytes": 594536164,\r\n      "sha256": "137291729dab9af257f0bfadc542f932c474ddd3bb3829d7e04f85935b38f00e",\r\n      "files": [\r\n        "data/processed/toys_games_full_temporal/interactions_test.csv.gz",\r\n        "data/processed/toys_games_full_temporal/interactions_train.csv.gz",\r\n        "data/processed/toys_games_full_temporal/interactions_validation.csv.gz",\r\n        "data/processed/toys_games_full_temporal/item_statistics.csv.gz",\r\n        "data/processed/toys_games_full_temporal/models/content_baseline/ranks_validation.npz",\r\n        "data/processed/toys_games_full_temporal/models/content_baseline_v2/metrics.json",\r\n        "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_test.npz",\r\n        "data/processed/toys_games_full_temporal/models/content_baseline_v2/ranks_validation.npz",\r\n        "data/processed/toys_games_full_temporal/split_manifest.json",\r\n        "data/processed/toys_games_full_temporal/test_with_history.csv.gz",\r\n        "data/processed/toys_games_full_temporal/train_with_history.csv.gz",\r\n        "data/processed/toys_games_full_temporal/validation_with_history.csv.gz",\r\n        "data/processed/ufm_positive_graph_h20_v1/complete.json",\r\n        "data/processed/ufm_positive_graph_h20_v1/positive_indptr.npy",\r\n        "data/processed/ufm_positive_graph_h20_v1/positive_items.npy",\r\n        "data/processed/ufm_positive_graph_h20_v1/user_keys.npy",\r\n        "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.csv.gz",\r\n        "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/samples_validation.npz",\r\n        "reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3/sampling_report.json"\r\n      ]\r\n    }\r\n  ],\r\n  "excluded": "Virtual environments, Git internals, secrets, temporary caches; no source files deleted.",\r\n  "logical_bytes": 3846478960,\r\n  "unique_bytes": 3846478960,\r\n  "completed_at_utc": "2026-10-29T00:09:27.297466+00:00"\r\n}\r\n'
EXPECTED_MANIFEST_SHA = '37691d8bc8a3e5e2acfb641394cfcd8da366eda6a987b7cc0584081de29bc6b1'
assert hashlib.sha256(BACKUP_MANIFEST_TEXT.encode()).hexdigest()==EXPECTED_MANIFEST_SHA
BACKUP_MANIFEST=json.loads(BACKUP_MANIFEST_TEXT)
INPUTS=Path('/content/ufm_verified_inputs'); INPUTS.mkdir(exist_ok=True)
(INPUTS/'manifest.json').write_text(BACKUP_MANIFEST_TEXT,encoding='utf-8')
DRIVE_INPUTS=DRIVE_ROOT/'inputs'; DRIVE_INPUTS.mkdir(exist_ok=True)
(DRIVE_INPUTS/'manifest.json').write_text(BACKUP_MANIFEST_TEXT,encoding='utf-8')
def copy_verified(src,dst,expected):
    src,dst=Path(src),Path(dst)
    if dst.is_file() and dst.stat().st_size==src.stat().st_size and sha(dst)==expected: return
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_name(dst.name+'.partial')
    shutil.copyfile(src,tmp)
    if sha(tmp)!=expected: raise RuntimeError('Copy SHA mismatch: '+str(tmp))
    tmp.replace(dst)

def workspace_verified(path):
    return all((path/e['path']).is_file() and (path/e['path']).stat().st_size==e['bytes']
               and sha(path/e['path'])==e['sha256'] for e in BACKUP_MANIFEST['files'])

if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
    if not workspace_verified(WORKSPACE):
        raise RuntimeError('Workspace exists but its SHA differs; refusing to overwrite.')
    print('REUSE_WORKSPACE_SHA_OK')
else:
    locations=[Path('/content'),Path('/content/colab_training_inputs_20261029'),
               Path('/content/ufm_upload'),DRIVE_INPUTS]
    pending={b['path']:b for b in BACKUP_MANIFEST['bundles']}
    while pending:
        for name,b in list(pending.items()):
            for folder in [INPUTS]+locations:
                candidate=folder/name
                if not candidate.is_file() or candidate.stat().st_size!=b['bytes']: continue
                if sha(candidate)!=b['sha256']:
                    raise RuntimeError('Archive size matches but SHA differs: '+str(candidate))
                copy_verified(candidate,INPUTS/name,b['sha256'])
                copy_verified(INPUTS/name,DRIVE_INPUTS/name,b['sha256'])
                print('ZIP_AND_DRIVE_SHA_OK',name,b['bytes'],flush=True)
                del pending[name]; break
        if pending:
            from google.colab import files
            import gc
            wanted = next(iter(pending))
            print('Còn thiếu:', ', '.join(pending), flush=True)
            print('Chọn MỘT tệp', wanted, 'trong thư mục colab_training_inputs_20261029 trên máy.', flush=True)
            previous_directory = Path.cwd()
            incoming = None
            try:
                os.chdir(INPUTS)
                incoming = files.upload()
                if not incoming:
                    raise RuntimeError('Chưa chọn ZIP. Gói đã nhận vẫn được giữ trên Drive; chạy lại cell để tiếp tục.')
                # files.upload may add a suffix when an interrupted upload exists.
                # Normalize only payloads matching a declared bundle exactly.
                for uploaded_name, payload in incoming.items():
                    matching = [b for b in pending.values()
                                if len(payload) == b['bytes']
                                and hashlib.sha256(payload).hexdigest() == b['sha256']]
                    if len(matching) != 1:
                        raise RuntimeError('Tệp tải lên không khớp ZIP còn thiếu: ' + uploaded_name)
                    target = INPUTS / matching[0]['path']
                    if not target.is_file() or target.stat().st_size != len(payload) or sha(target) != matching[0]['sha256']:
                        temporary = target.with_name(target.name + '.partial')
                        temporary.write_bytes(payload)
                        temporary.replace(target)
                del payload
            finally:
                os.chdir(previous_directory)
                if incoming is not None:
                    incoming.clear()
                gc.collect()
    subprocess.run([sys.executable,str(SOURCE/'scripts/artifact_backup.py'),'restore',
                    '--backup',str(INPUTS),'--output',str(WORKSPACE)],check=True)
    if not workspace_verified(WORKSPACE): raise RuntimeError('Restored file SHA mismatch')
receipt={'scope':'actual_colab_runtime','files':len(BACKUP_MANIFEST['files']),
         'manifest_sha256':EXPECTED_MANIFEST_SHA,'all_files_sha256_verified':True,
         'source_commit':SOURCE_COMMIT,'gpu':gpu_line,'workspace':str(WORKSPACE),
         'verified_at_utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()}
save_json(DRIVE_ROOT/'colab_input_receipt.json',receipt)
spec=importlib.util.spec_from_file_location('locked_runner',SOURCE/'scripts/run_locked_ufm_followup.py')
locked=importlib.util.module_from_spec(spec); spec.loader.exec_module(locked)
PROTOCOL=json.loads((SOURCE/'configs/multimodal_followup_20261028.json').read_text())
CHECKED=locked.preflight(PROTOCOL,SOURCE,WORKSPACE)
print('COLAB_PREFLIGHT_PASSED',len(CHECKED),'locked hashes; 15 jobs; no test evaluation')

## Bộ chạy nối tiếp và sao lưu

Train từng đoạn 1.000 update với cùng loss, seed, ngân sách và quy tắc chọn checkpoint đã khóa. Mỗi đoạn kết thúc phải sao lưu Drive và đối chiếu SHA trước khi tiếp tục. Cấu hình/source/phiên bản khác khi resume bị từ chối. Metric validation giữ trong log, chỉ tổng hợp bảng khi đủ bằng chứng.
Khi mở phiên mới, kết quả đánh giá đã sao lưu được khôi phục từ Drive sau khi đối chiếu SHA; không cần đánh giá lại các lượt đã đủ report/predictions.


In [ ]:
import re, gc, zipfile, statistics
RUNS=SOURCE/'runs/ufm_followup_20261028'
REPORTS=SOURCE/'reports/ufm_followup_20261028'
RUNS.mkdir(parents=True,exist_ok=True); REPORTS.mkdir(parents=True,exist_ok=True)
JOBS=[(stage['name'],variant,seed) for stage in PROTOCOL['stages'] for variant in stage['variants'] for seed in PROTOCOL['seeds']]

def sync_folder(local,remote):
    local,remote=Path(local),Path(remote); remote.mkdir(parents=True,exist_ok=True)
    entries=[]
    for p in sorted(local.rglob('*')):
        if not p.is_file() or p.name.endswith(('.tmp','.partial')): continue
        rel=p.relative_to(local).as_posix(); digest=sha(p)
        copy_verified(p,remote/rel,digest)
        entries.append({'path':rel,'bytes':p.stat().st_size,'sha256':digest})
    save_json(remote/'sync_receipt.json',{'files':entries,'source_commit':SOURCE_COMMIT,
        'verified_at_utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()})
    print('DRIVE_CHECKPOINT_SHA_OK',local.name,len(entries),'files',flush=True)

def restore_run(label):
    dest=RUNS/label; remote=DRIVE_ROOT/'runs'/label
    receipt=remote/'sync_receipt.json'
    if (dest/'latest.pt').is_file() or (dest/'completed.json').is_file(): return dest
    if dest.exists() and any(dest.iterdir()): raise RuntimeError('Partial local run has no latest.pt: '+str(dest))
    if receipt.is_file():
        record=json.loads(receipt.read_text())
        for entry in record['files']:
            rel=Path(entry['path'])
            if rel.is_absolute() or '..' in rel.parts: raise ValueError('Unsafe resume path')
            source=remote/rel
            if source.stat().st_size!=entry['bytes'] or sha(source)!=entry['sha256']:
                raise RuntimeError('Drive resume SHA mismatch: '+str(source))
            copy_verified(source,dest/rel,entry['sha256'])
        print('RESUME_FROM_DRIVE_SHA_OK',label)
    return dest

def restore_reports():
    remote = DRIVE_ROOT/'reports'
    receipt = remote/'sync_receipt.json'
    if not receipt.is_file():
        if remote.exists() and any(remote.iterdir()):
            raise RuntimeError('Drive reports have no sync receipt; audit the partial backup before continuing.')
        return
    record = json.loads(receipt.read_text(encoding='utf-8'))
    if record.get('source_commit') != SOURCE_COMMIT:
        raise RuntimeError('Drive reports source commit mismatch')
    for entry in record['files']:
        rel = Path(entry['path'])
        if rel.is_absolute() or '..' in rel.parts:
            raise ValueError('Unsafe report restore path')
        source = remote/rel
        if (not source.is_file() or source.stat().st_size != entry['bytes']
                or sha(source) != entry['sha256']):
            raise RuntimeError('Drive report SHA mismatch: '+str(source))
    # Verify the complete backup before restoring any report.
    for entry in record['files']:
        rel = Path(entry['path'])
        copy_verified(remote/rel, REPORTS/rel, entry['sha256'])
    print('REPORTS_FROM_DRIVE_SHA_OK',len(record['files']),'files',flush=True)

def cursor(output):
    if not (output/'latest.pt').is_file(): return 0
    import torch
    state=torch.load(output/'latest.pt',map_location='cpu',weights_only=True)
    value=int(state['global_step']); del state; gc.collect(); return value

def run_logged(command,log_path):
    log_path.parent.mkdir(parents=True,exist_ok=True)
    observations=[]; tail=[]
    with log_path.open('a',encoding='utf-8') as logfile:
        process=subprocess.Popen(command,cwd=SOURCE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,bufsize=1)
        try:
            for line in process.stdout:
                logfile.write(line); logfile.flush(); tail=(tail+[line])[-25:]
                match=re.search(r'ufm_epoch=(\d+) batch=(\d+)/(\d+) step=(\d+).*vram_GiB=([\d.]+)',line)
                if match:
                    observations.append({'at':time.monotonic(),'step':int(match[4]),'batches':int(match[3]),'vram_gib':float(match[5])})
                    print(line.strip(),flush=True)
                elif line.startswith(('UFM TRAIN START','TRAINABLE PARAMS','RESUME PASS','CONTROLLED INTERRUPT','UFM TRAIN COMPLETE')):
                    print(line.strip(),flush=True)
            status=process.wait()
        except BaseException:
            process.terminate(); process.wait(); raise
    if status: raise RuntimeError('Training/evaluation failed:\n'+''.join(tail))
    return observations

def sync_run(label):
    output=RUNS/label
    if output.exists(): sync_folder(output,DRIVE_ROOT/'runs'/label)
    log=RUNS/f'{label}.log'
    if log.exists(): copy_verified(log,DRIVE_ROOT/'logs'/log.name,sha(log))

def train_slice(variant,seed,stop_step):
    label=f'{variant}_s{seed}'; output=restore_run(label)
    if (output/'completed.json').is_file():
        locked.verify_completed(output,variant,seed,PROTOCOL); return []
    if not locked.gpu_idle(): raise RuntimeError('GPU busy with another process; no process was stopped.')
    command=locked.command_for(PROTOCOL,SOURCE,WORKSPACE,variant,seed,output,(output/'latest.pt').is_file())
    command+=['--interrupt-after-step',str(stop_step)]
    try: observations=run_logged(command,RUNS/f'{label}.log')
    finally:
        if (output/'latest.pt').is_file(): sync_run(label)
    return observations

def evaluate_complete(variant,seed):
    label=f'{variant}_s{seed}'; output=RUNS/label
    locked.verify_completed(output,variant,seed,PROTOCOL)
    report=REPORTS/f'{label}_v3.json'; predictions=REPORTS/f'{label}_v3_scores'
    if report.exists():
        if not (predictions/'scores_validation.npz').is_file(): raise RuntimeError('Partial evaluation: '+label)
    else:
        if predictions.exists(): raise RuntimeError('Partial prediction output: '+label)
        cmd=[sys.executable,'-u',str(SOURCE/'src/reevaluate_ufm_checkpoints.py'),
             '--checkpoint',str(output/'best.pt'),'--evaluation',str(WORKSPACE/'reports/fix_1_15_20261027/fitlab01_finish1/evaluation_v3'),
             '--features',str(WORKSPACE/'data/processed/toys_games_full_temporal/foundation_clip_b32_v1'),
             '--counts',str(WORKSPACE/'data/processed/gpu_cache_h20_t64_v1/train_counts.npy'),
             '--output',str(report),'--save-predictions',str(predictions),'--device','cuda','--splits','validation']
        run_logged(cmd,REPORTS/f'{label}_v3.log')
    sync_folder(REPORTS,DRIVE_ROOT/'reports'); sync_run(label)
    print('RUN_COMPLETE_AND_BACKED_UP',label,flush=True)

def export_label(label):
    # Include optimizer/RNG for resume; local download needs its own SHA verification.
    out=SOURCE/'exports'/label; out.mkdir(parents=True,exist_ok=True)
    snapshot=out/f'{label}_step{cursor(RUNS/label)}.zip'
    members=[p for p in (RUNS/label).rglob('*') if p.is_file() and not p.name.endswith(('.tmp','.partial'))]
    members += [p for p in REPORTS.rglob('*') if p.is_file() and p.relative_to(REPORTS).parts[0].startswith(label)]
    members += [p for p in (RUNS/f'{label}.log',SOURCE/'configs/multimodal_followup_20261028.json',SOURCE/'src/train_ufm_recommender.py',SOURCE/'src/ufm_model.py',SOURCE/'src/train_gpu_recommender.py') if p.is_file()]
    if not snapshot.exists():
        temp=snapshot.with_suffix('.zip.partial')
        with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as archive:
            for p in members: archive.write(p,p.relative_to(SOURCE).as_posix())
        temp.replace(snapshot)
    record={'path':str(snapshot),'bytes':snapshot.stat().st_size,'sha256':sha(snapshot),
            'label':label,'step':cursor(RUNS/label),'local_download_verified':False}
    save_json(snapshot.with_suffix('.receipt.json'),record)
    save_json(DRIVE_ROOT/'exports'/label/snapshot.with_suffix('.receipt.json').name,record)
    print('CHECKPOINT_EXPORT_READY',json.dumps(record),flush=True)
    return record
restore_reports()
print('ORCHESTRATOR_READY',len(JOBS),'locked jobs')

## Bắt đầu huấn luyện thật: full UFM / seed 42

100 update đầu thuộc chính lượt huấn luyện, được lưu rồi resume tiếp. Notebook dùng tốc độ quan sát được để phân bổ thời gian phiên; không chạy dữ liệu giả và không giảm ngân sách thực nghiệm.


In [ ]:
first=restore_run('full_s42')
existing=cursor(first)
benchmark_path=DRIVE_ROOT/f'{GPU_TYPE.lower()}_throughput.json'
if not benchmark_path.exists() and not (first/'completed.json').exists():
    stop_step = PROBE_STEPS if existing == 0 else existing + 2 * PROBE_STEPS
    observations=train_slice('full',42,stop_step)
    if len(observations)<2: raise RuntimeError('Chưa đủ hai mốc để đo throughput')
    a,b=observations[0],observations[-1]
    seconds_per_step=(b['at']-a['at'])/(b['step']-a['step'])
    max_total_steps=b['batches']*PROTOCOL['training']['epochs']*len(JOBS)
    benchmark={'gpu':gpu_line,'source_commit':SOURCE_COMMIT,'measured_steps':b['step']-a['step'],
        'seconds_per_step':seconds_per_step,'updates_per_second':1/seconds_per_step,
        'examples_per_second':PROTOCOL['training']['batch_size']/seconds_per_step,
        'observed_peak_vram_gib':max(x['vram_gib'] for x in observations),
        'max_steps_all_15_without_early_stop':max_total_steps,
        'estimated_upper_budget_gpu_hours_excluding_validation_and_io':seconds_per_step*max_total_steps/3600,
        'full_training_completed':False,'scope':'real_training_prefix_resumable_not_ranking_result'}
    save_json(benchmark_path,benchmark)
elif benchmark_path.exists(): benchmark=json.loads(benchmark_path.read_text())
else: raise RuntimeError('Checkpoint exists but the saved throughput measurement is absent.')
print(json.dumps(benchmark,ensure_ascii=False,indent=2))
print('Tiếp tục full_s42 từ checkpoint vừa lưu; chưa có kết quả xếp hạng cuối.', flush=True)


## Tiếp tục train và sao lưu checkpoint lên Drive

Ưu tiên lần lượt full UFM, semantic-only và fixed-fusion trên seed 42, 7, 2026, sau đó hai cải tiến. Phiên này dùng ngân sách 120 phút tính từ cell đầu, bao gồm chuẩn bị dữ liệu; các đoạn train có khoảng dự phòng để sao lưu. Colab vẫn có thể ngắt sớm, nên giữ checkpoint từ các đoạn đã sao lưu. Phiên sau chạy lại cùng notebook để resume; giữ cùng phiên bản runtime/PyTorch khi Colab cho phép.


In [ ]:
last_label=None
for stage,variant,seed in JOBS:
    label=f'{variant}_s{seed}'; output=restore_run(label)
    if stage!='controls':
        for v in PROTOCOL['stages'][0]['variants']:
            for s in PROTOCOL['seeds']:
                control=restore_run(f'{v}_s{s}')
                locked.verify_completed(control,v,s,PROTOCOL)
                if not (REPORTS/f'{v}_s{s}_v3.json').exists():
                    remote_report=DRIVE_ROOT/'reports'/f'{v}_s{s}_v3.json'
                    if not remote_report.exists(): raise RuntimeError('Chưa đủ 9 đối chứng đã đánh giá; chưa chạy cải tiến.')
    while not (output/'completed.json').exists():
        elapsed=(time.monotonic()-SESSION_T0)/60
        next_slice_minutes=SLICE_STEPS*benchmark['seconds_per_step']/60 + 10
        if elapsed+next_slice_minutes > SESSION_BUDGET_MINUTES:
            print('SESSION_PAUSED_KEEP_FULL_PROTOCOL',label,'step',cursor(output))
            last_label=label if (output/'latest.pt').exists() else last_label
            break
        step=cursor(output)
        train_slice(variant,seed,step+SLICE_STEPS)
        last_label=label
    if not (output/'completed.json').exists(): break
    evaluate_complete(variant,seed)
    last_label=label
if last_label: export_label(last_label)
print('Đã sao lưu checkpoint tạo được; chỉ run có completed.json hợp lệ mới tính hoàn tất.')

## Tiến độ và đường dẫn kết quả

Checkpoint, config, log và kết quả đã sinh được lưu tại:
`MyDrive/UFM-Rec_Experiments/ufm_followup_20261028`

Khi hiện `DRIVE_CHECKPOINT_SHA_OK`, bản sao của đoạn vừa chạy đã được đối chiếu. Có thể mở thanh **Tệp** bên trái, vào `/content/UFM-Rec/exports/<label>`, nhấp phải ZIP và receipt để tải về máy. ZIP chỉ được tạo ở cuối phần chạy, không tạo lại sau mỗi 100 update. Lần sau mở cùng notebook và chạy lại để nối tiếp từ bản Drive.

Chỉ tổng hợp khi đủ 15 lượt; không coi checkpoint giữa epoch là mô hình đã nghiệm thu.


In [ ]:
completed=[]
for stage,variant,seed in JOBS:
    label=f'{variant}_s{seed}'
    restore_run(label)
    if (RUNS/label/'completed.json').is_file():
        locked.verify_completed(RUNS/label,variant,seed,PROTOCOL); completed.append(label)
print('VERIFIED_COMPLETED',len(completed),'/ 15',completed)
if all((REPORTS/f'{v}_s{s}_v3.json').is_file() for _,v,s in JOBS):
    summary=REPORTS/'summary_all_seeds.json'
    if not summary.exists():
        subprocess.run([sys.executable,str(SOURCE/'scripts/summarize_locked_ufm_followup.py'),
                        '--reports',str(REPORTS),'--output',str(summary)],check=True)
    sync_folder(REPORTS,DRIVE_ROOT/'reports')
    print(summary.read_text())
else:
    print('Chưa đủ 15 report; chưa xuất kết luận cải tiến hoặc mean/std thiếu seed.')
print('DRIVE_BACKUP',DRIVE_ROOT)
